# Õppetund 10 - AI agentid tootmises

Selles õppetükis õpid AI agentide **tootmismustreid** Microsoft Agent Frameworki (Python) abil. Katame:

- **Jälgitavus** — ajastuse ja logimise lisamine agendi interaktsioonidele
- **Hindamine** — hindamisagendi kasutamine vastuse kvaliteedi hindamiseks
- **Kulukorraldus** — strateegiad tokenite optimeerimiseks ja mudeli valikuks

Stsenaariumiks on **reisibüroo**, mis aitab kasutajatel reisiplaane teha, millele on lisatud jälgimine ja hindamine.


## Seadistamine


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Tootmisalased kaalutlused

Tehisintellekti agentide viimine prototüüpidest tootmisse nõuab hoolikat tähelepanu kolme sambale:

1. **Jälgitavus** — Sul peab olema ülevaade sellest, mida agent teeb, kui kaua see aega võtab ja milliseid tööriistu kasutab. Ilma jälgimise ja logimiseta on tootmisprobleemide tõrkeotsing peaaegu võimatu.

2. **Hindamine** — Automatiseeritud kvaliteedikontrollid tagavad, et agendi vastused jäävad aja jooksul täpsed, põhjalikud ja kasulikud. Hindaja-agent saab vastuseid skoorida määratletud kriteeriumite alusel.

3. **Kulujuhtimine** — Sümbolite kasutamine mõjutab otseselt kulusid. Sellised strateegiad nagu sisendi optimeerimine, mudeli valik ja vahemällu salvestamine aitavad kulusid kontrolli all hoida ilma kvaliteedis järeleandmisi tegemata.


## Vaatleva Agendi Loomine

Me määratleme reisitööriistad ja ühendame agendi kutse ajastamisega, et saaksime jälgida latentsust. Tootmiskeskkonnas integreeriksite selle OpenTelemetry või sarnase jälgimissüsteemiga.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Evaluation Patterns

A common production pattern is to use a second agent as an **evaluator**. The evaluator scores the primary agent's response against predefined criteria such as completeness, accuracy, and helpfulness.

This enables:
- Automated quality gates before responses reach users
- Regression detection when prompts or models change
- Continuous monitoring of agent performance over time


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Kulujuhtimise strateegiad

Kulude kontrollimine on tootmis-IA agentide jaoks kriitiline. Siin on peamised strateegiad:

| Strateegia | Kirjeldus |
|---|---|
| **Käskude optimeerimine** | Hoia süsteemi juhised lühikesed. Eemalda liigsed kontekstid sisendikaundite vähendamiseks. |
    "| **Mudeli valik** | Kasuta lihtsateks ülesanneteks nagu klassifitseerimine või andmete väljavõtmine väiksemaid ja odavamaid mudeleid (nt GPT-4.1-mini) ning jäta keerulisemad päringud võimsamatele mudelitele. |\n",
| **Vahemällu salvestamine** | Salvesta tööriistade tulemusi ja sagedasi päringuid vahemällu, et vältida korduvaid API kõnesid. |
| **Tokeni eelarved** | Sea `max_tokens` piiranguid, et vältida ootamatult pikki vastuseid. |
| **Pakkimine** | Võimalusel grupeerida mitu kasutajapäringut ühte API kõnesse. |

Praktikas töötab hästi kihiline lähenemine: suuna lihtsad päringud kiiresti ja odavalt töötlevasse mudelisse ning keerulisemad päringud edasta võimekamale mudelile.


## Kokkuvõte

Selles õppetükis õppisite, kuidas:

1. **Lisada vaatlusvõime** agentide interaktsioonidele ajastuse ja logimise abil, luues aluse jälgimiseks ja monitoorimiseks.
2. **Automaatselt hinnata agentide vastuseid** hindamisagendi abil, mis skoorib vastuste täielikkust, täpsust ja kasulikkust.
3. **Kulusid hallata** kaudu kiire optimeerimise, mudeli valiku, vahemällu salvestamise ja tokeni eelarvestuse.

Need tootmismustrid aitavad tagada, et teie tehisintellekti agendid on usaldusväärsed, mõõdetavad ja kulutõhusad suures mahus.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
